In [27]:
import os
import glob
from ultralytics import YOLO
import yaml

def run_full_pipeline():
    """
    Trains, validates, and tests a YOLOv8 model. Then, it classifies
    test images based on Q-block count and calculates the
    accuracy of this classification logic against file-name-based ground truth.
    """
    config_path = 'datasets/eagle-eyes-4/data.yaml'

    # --- Pre-run Checks ---
    if not os.path.exists(config_path):
        print(f"Error: Configuration file not found at '{config_path}'")
        return

    # --- 1. Load a Pre-trained YOLOv8 Model ---
    print("Loading pre-trained YOLOv8n model...")
    model = YOLO('yolov8n.pt')

    # --- 2. Train the Model on Your Custom Data ---
    print(f"Starting training with configuration: {config_path}")
    results = model.train(
        data=config_path,
        epochs=15,
        imgsz=640,
        batch=-1,
        name='yolov8n_qblock_detector'
    )
    
    print("--- Training complete! ---")
    
    # --- 3. Display Final Validation Metrics ---
    print("\nFinal Validation Metrics from Training:")
    print(f"  mAP50-95: {results.box.map:.4f}")
    print(f"  Precision: {results.box.mp:.4f}")
    print(f"  Recall: {results.box.mr:.4f}")

    # --- 4. Load the Best Model for Inference ---
    print("\nLoading the best model for inference...")
    results_directory = results.save_dir
    best_model_path = os.path.join(results_directory, 'weights/best.pt')
    if not os.path.exists(best_model_path):
        print(f"Error: Could not find best model at '{best_model_path}'.")
        return
    
    best_model = YOLO(best_model_path)
    print(f"Successfully loaded model from: {best_model_path}")

    # --- 5. Test Model and Apply Business Logic ---
    print(f"\n--- Starting Test & QC Phase ---")
    
    with open(config_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    if 'test' not in data_yaml or not data_yaml['test']:
        print("Warning: 'test' path not found in your YAML file. Skipping test phase.")
        return    

    base_dir = os.path.dirname(os.path.abspath(config_path))
   
    test_img_dir = base_dir + "/test/images"
    
    print(f"BASE DIRECTORY:", base_dir)
    print(f"TEST IMAGE DIRECTORY:" ,test_img_dir)

    if not os.path.exists(test_img_dir):
        print(f"Error: Test directory specified in YAML does not exist: '{test_img_dir}'")
        return

    image_extensions = ['*.bmp', '*.jpg', '*.jpeg', '*.png']
    test_images = []
    for ext in image_extensions:
        test_images.extend(glob.glob(os.path.join(test_img_dir, ext)))

    if not test_images:
        print(f"Warning: No images found in the test directory '{test_img_dir}'. Skipping test phase.")
    else:
        print(f"Found {len(test_images)} images to test from path specified in YAML.")
        
        # --- QC LOGIC & ACCURACY CALCULATION ADDED HERE ---
        
        # 1. DEFINE YOUR "GOOD" STATE:
        # Based on your "OK" images, a full sheet has 3 rows * 6 tickets/row * 2 Q-blocks/ticket = 36
        # *** IMPORTANT: CHANGE THIS NUMBER IF YOUR IMAGES SHOW A DIFFERENT AMOUNT (e.g., 2) ***
        EXPECTED_QBLOCK_COUNT = 14
        print(f"Applying QC Rule: Image is 'GOOD' if Q-block count is exactly {EXPECTED_QBLOCK_COUNT}.")

        # 2. Run prediction. This saves images AND returns results.
        predict_results = best_model.predict(
            source=test_images, 
            save=True, 
            project="test_results",
            name="predictions_with_qc_accuracy", 
            exist_ok=True,
            conf=0.5 # Set a confidence threshold
        )
        
        print("\n--- QC Classification Results ---")
        
        correct_predictions = 0
        total_images_processed = 0
        
        # 3. Loop through results and classify each image
        for image_path, result in zip(test_images, predict_results):
            image_name = os.path.basename(image_path)
            
            # --- Determine Ground Truth (from filename) ---
            ground_truth = "UNKNOWN"
            if "_OK" in image_name.upper():
                ground_truth = "GOOD"
            elif "_NG" in image_name.upper():
                ground_truth = "NOT_GOOD"
            
            # --- Determine Model Prediction (from Q-block count) ---
            num_detections = len(result.boxes)
            model_prediction = "GOOD" if num_detections == EXPECTED_QBLOCK_COUNT else "NOT_GOOD"

            # --- Score the Prediction ---
            if ground_truth == "UNKNOWN":
                print(f"Image: {image_name:<45} | GT: {ground_truth:<9} | Pred: {model_prediction:<9} ({num_detections} blocks) | Result: [SKIPPED]")
                continue
            
            total_images_processed += 1
            
            if model_prediction == ground_truth:
                correct_predictions += 1
                status = "[CORRECT]"
            else:
                status = "[INCORRECT]"

            print(f"Image: {image_name:<45} | GT: {ground_truth:<9} | Pred: {model_prediction:<9} ({num_detections} blocks) | Result: {status}")

        print("\n--- Testing Complete! ---")
        print("Predicted test images are saved in 'test_results/predictions_with_qc_accuracy'")

        # --- 4. Calculate and Print Final Accuracy ---
        if total_images_processed > 0:
            accuracy = (correct_predictions / total_images_processed) * 100
            print("\n--- Final QC Model Accuracy ---")
            print(f"Total Test Images Scored: {total_images_processed}")
            print(f"Correct Predictions:    {correct_predictions}")
            print(f"QC Model Accuracy:        {accuracy:.2f}%")
        else:
            print("\nNo scorable images (containing '_OK' or '_NG') were found in the test set.")


if __name__ == '__main__':
    run_full_pipeline()


Loading pre-trained YOLOv8n model...
Starting training with configuration: datasets/eagle-eyes-4/data.yaml
New https://pypi.org/project/ultralytics/8.3.223 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.111 🚀 Python-3.12.4 torch-2.6.0 CPU (Apple M3)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=datasets/eagle-eyes-4/data.yaml, epochs=1, time=None, patience=100, batch=-1, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=yolov8n_qblock_detector45, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=Fal

train: Scanning /Users/vishali/Documents/CDA-500/prototype/datasets/eagle-eyes-4/train/labels.cache... 834 images, 0 backgrounds, 0 corrupt: 100%|██████████| 834/834 [00:00<?, ?it/s]

AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch:  ⚠️ intended for CUDA devices, using default batch-size 16
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 258.6±57.8 MB/s, size: 80.6 KB)



train: Scanning /Users/vishali/Documents/CDA-500/prototype/datasets/eagle-eyes-4/train/labels.cache... 834 images, 0 backgrounds, 0 corrupt: 100%|██████████| 834/834 [00:00<?, ?it/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 275.9±126.9 MB/s, size: 72.0 KB)


...

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/yolov8n_qblock_detector45
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/1         0G      1.668      2.371     0.8237         84        640: 100%|██████████| 53/53 [05:22<00:00,  6.08s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:12<00:00,  4.05s/it]


                   all         79       2551      0.081      0.492      0.464      0.296

1 epochs completed in 0.093 hours.
Optimizer stripped from runs/detect/yolov8n_qblock_detector45/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/yolov8n_qblock_detector45/weights/best.pt, 6.2MB

Validating runs/detect/yolov8n_qblock_detector45/weights/best.pt...
Ultralytics 8.3.111 🚀 Python-3.12.4 torch-2.6.0 CPU (Apple M3)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:10<00:00,  3.63s/it]


                   all         79       2551     0.0809      0.491      0.464      0.296
     backgroundclutter         27         33          0          0          0          0
      incorrect qblock          4         12          0          0          0          0
                qblock         79       1077      0.127      0.995       0.99      0.687
           smallqblock         79       1429      0.197       0.97      0.866      0.496
Speed: 1.1ms preprocess, 129.4ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/yolov8n_qblock_detector45
--- Training complete! ---

Final Validation Metrics from Training:
  mAP50-95: 0.2956
  Precision: 0.0809
  Recall: 0.4913

Loading the best model for inference...
Successfully loaded model from: runs/detect/yolov8n_qblock_detector45/weights/best.pt

--- Starting Test & QC Phase ---
BASE DIRECTORY: /Users/vishali/Documents/CDA-500/prototype/datasets/eagle-eyes-4
TEST IMAGE DIRECTORY: /Users/vishali/Documents/CDA